##### 导包

In [2]:
from dotenv import load_dotenv
load_dotenv()
from pydantic import BaseModel,Field
from typing import Literal
from langchain_core.tools import tool
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.agents import create_agent
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
from datetime import datetime,timedelta
from langchain.messages import HumanMessage
import re

C:\Users\15162\AppData\Local\Temp\ipykernel_22692\2311909746.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


##### 复制项目代码

In [3]:
# 初始化embedding模型
emd = HuggingFaceEmbeddings(
    model_name = "D:/AI_tool/embedding_models/BAAI/models/BAAI--bge-base-zh-v1.5/snapshots/master",
    model_kwargs={"device": "cpu"}
)

# 加载数据库
vector_db = Chroma(
    persist_directory = "C:/Users/15162/Desktop/VS/item/agent/local_doc_db",
    embedding_function=emd
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

C:\Users\15162\AppData\Local\Temp\ipykernel_22692\1860905620.py:8: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_db = Chroma(


In [4]:
# 计算信任度的函数
def confidance(respond):
    if not respond:
        return 0.0

    scores = [score for _, score in respond]
    # 平均分
    avg_score = sum(scores)/len(scores)
    # 最高分
    max_score = max(scores)
    # 一致性
    score_range = max(scores) - min(scores)
    consistency = 1.0 - min(score_range / 0.3, 1.0)

    # 得到信任度
    confidance = avg_score * 0.4 + max_score * 0.3 + consistency * 0.3

    confidance = min(confidance,1.0)

    return confidance

# 脏数据处理函数
def clear_data(respond:str):
    lines = respond.split('\n')
    keep_lines = []
    for line in lines:
        if any(kw in line for kw in ['邮购电话', '质量投诉', '盗版侵权', 
            '咨询联系方式', '版权所有', '翻印必究',
            '联系及邮购','邮箱']):
            continue
        keep_lines.append(line)
    respond = '\n'.join(keep_lines)

    # 删除qq号、qq群号：
    pattern=r'(?:QQ|qq)[\u4e00-\u9fa5]*\s*[: ：]\s*[1-9]\d{3,12}'
    respond = re.sub(pattern, '', respond, flags=re.IGNORECASE)
    # 删除页数，章数
    pattern=r'[第]\s*\d+\s*[页|章|版]'
    respond = re.sub(pattern, '', respond, flags=re.IGNORECASE)
    # 删除邮箱
    pattern=r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'
    respond = re.sub(pattern, '', respond, flags=re.IGNORECASE)
    # 删除空白
    respond = re.sub(r'\s+', ' ', respond)

    respond = respond.strip()
    # respond = ''.join(respond)
    return respond

In [5]:

class NoteInput(BaseModel):
    user_query:str = Field(description="用户提问的问题")

@tool
def Get_Notes(user_query:str):
    '''
    - user_query:用户提问的问题
    - 每次用户提问时都检查一下回复的信任度，若信任度过低，则让用户更换询问方式
    - 每次回答都先给出信任度后在做回答，确保尽可能高的信任度
    '''
    # 检索
    try:
        if user_query is None:
            print("请输入您的问题")
        # 计算信任度
        count = 0
        while count < 3:
            respond = vector_db.similarity_search_with_score(user_query,k=2)
            # 检查检索结果
            if not respond or len(respond) == 0:
                return "未找到相关信息，请换个方式提问"
            confidant = confidance(respond)# 第一次计算
            if confidant >= 0.7:#置信度大于70%就无需继续检索
                break
            else:
                count += 1
        # 处理脏数据
        raw_content = respond[0][0].page_content
        clear_respond = clear_data(raw_content)
        # clear_respond = 1
    
        if confidant >= 0.7:
            result= f"置信度是{confidant * 100}%,可以相信,内容是{clear_respond}"
        elif confidant >= 0.4:
            result= f"置信度是{confidant * 100}%,置信度较低,请谨慎相信此答案,内容是{clear_respond}"
        else:
            result= f"置信度是{confidant * 100}%,无法找到答案，请换个询问方式"

        return result
    except Exception as e:
        return f"检索失败:{e}"



In [6]:
# 这里指定记忆缓存的位置
connection = sqlite3.connect("./test_SQlite/checkpoint.db",check_same_thread=False)
checkpointer = SqliteSaver(connection)# 初始化checkpointer
checkpointer.setup()

config = {"configurable":{"thread_id": "thread_2"}}

# 自动清理函数
def clean_thread(thread_id_to_delete):
    """删除特定会话的所有历史"""
    try:
        cursor = connection.cursor()
        cursor.execute(
            "DELETE FROM checkpoints WHERE thread_id = ?",
            (thread_id_to_delete,)
        )
        cursor.execute(
            "DELETE FROM writes WHERE thread_id = ?",  # 同时删除关联的 writes 表
            (thread_id_to_delete,)
        )
        connection.commit()
        print(f"已删除 thread_id: {thread_id_to_delete}")
    except Exception as e:
        print(f"删除失败: {e}")

In [7]:
# 创建agent
agent = create_agent(
    model="deepseek-v4-pro",
    system_prompt='''你是一个学习助手，帮助用户查找知识库中的知识点。使用 Get_Notes 工具检索，根据置信度回答。输出使用 Markdown 格式''',
    checkpointer=checkpointer,
    # middleware=[middleware],
    tools=[Get_Notes]
)

##### 测试模块

In [8]:
# ========================信任度检测========================
def test_confidant():
    # 实验用户问题
    test_query = "什么是计算机，计算机发展了多少年"

    # 模拟查询
    print("=" * 25 + "模拟查询" + "=" * 25)
    mock_results = [# 模拟索引
        (type('Doc', (), {'page_content': 'CPU是中央处理器'})(), 0.85),
        (type('Doc', (), {'page_content': 'CPU负责执行指令'})(), 0.72),
        (type('Doc', (), {'page_content': 'CPU是计算机核心'})(), 0.61)
]

    confidant = confidance(mock_results)

    if 0 <= confidant <= 1:
        print(f"模拟信任度结果{confidant * 100}%,模拟结果合理\n")
    else:
        print("模拟结果不在合理范围内\n")

    # 检测空数据
    empty_confidant = confidance([])
    if empty_confidant == 0.0:
        print("空查询信任度0%,结果正确\n")
    else:
        print(f"空查询信任度应为0%,但真实结果为{empty_confidant * 100}%\n")

# 真实查询
    print("=" * 25 + "真实查询" + "=" * 25)
    response = agent.invoke({"messages":[HumanMessage(test_query)]},config)
    print(f"AI:{response['messages'][-1].content}\n")

# 运行测试函数
test_confidant()

=========================模拟查询=========================
模拟信任度结果60.566666666666656%,模拟结果合理

空查询信任度0%,结果正确

=========================真实查询=========================
AI:## 检索结果说明

### 问题一：什么是计算机

**信任度：76.50%**（可以相信）

⚠️ 需要说明的是：本次检索返回的内容实际上是关于**"计算机网络的定义"**，而非"计算机"本身，内容与您的问题存在一定偏差。检索到的相关定义如下，供参考：

> 计算机网络主要是由一些通用的、可编程的硬件互连而成的，而这些硬件并非专门用来实现某一特定目的（例如，传送数据或视频信号）。这些可编程的硬件能够用来传送多种不同类型的数据，并能支持广泛的和日益增长的应用。

根据该定义：
- 计算机网络所连接的硬件并不限于一般的计算机，还包括智能手机
- 计算机网络并非专门用来传送数据，而是能够支持很多种应用
- 其中的"可编程硬件"一定包含**中央处理机 CPU**

> 💡 如果您想了解"计算机"本身的概念，建议更换问法，例如："什么是电子计算机"或"计算机由哪些部分组成"。

---

### 问题二：计算机发展了多少年

**信任度：86.66%**（可以相信）

⚠️ 需要说明的是：本次检索返回的内容为教材的出版历史（1989 年首次出版，历经 1994、1999、2003、2008、2013 年等多次修订），**并未直接回答"计算机发展了多少年"**，因此该答案与您的问题不匹配。

> 💡 建议您更换询问方式，例如："计算机的发展历史"、"计算机的发展阶段"或"第一台计算机诞生于哪一年"，以便获取更准确的答案。



In [8]:
# ========================脏数据处理========================
sample_1 = """
邮购电话：（010）88254888
CPU是中央处理器
邮箱：zlts@phei.com.cn
"""

sample_2 = """
第291页
网络层负责数据包转发
① 这是脚注内容
"""

sample_3 = """
QQ交流群：123456789
QQ:12345567
操作系统是管理计算机硬件和软件的程序
版权所有 翻印必究
"""

sample_4 = """
联系及邮购电话：（010）88254888
计算机网络（第7版）谢希仁
质量投诉请发邮件至 zlts@phei.com.cn
本书咨询联系方式：QQ 9616328
第3章 网络体系结构
"""
def test_clean_function():
    # 整合脏数据样本
    sample = [sample_1,sample_2,sample_3,sample_4]
    # 开始清理脏数据
    for i,dirty in enumerate(sample,1):
        print("=" * 20 + f"进行第{i}段清理" + "=" * 20)

        clear_sample = clear_data(dirty)

        print(f"清理完后的样本：{clear_sample}")

# 运行测试脏数据处理
test_clean_function()

====================进行第1段清理====================
清理完后的样本：CPU是中央处理器
====================进行第2段清理====================
清理完后的样本：网络层负责数据包转发 ① 这是脚注内容
====================进行第3段清理====================
清理完后的样本：操作系统是管理计算机硬件和软件的程序
====================进行第4段清理====================
清理完后的样本：计算机网络（）谢希仁 网络体系结构


In [9]:
# ========================异常检测========================
def test_error():
    test_cases = [{
            "name": "空查询",
            "query": "",
            "expected": ["未找到", "请输入", "不能为空"]
        },
        {
            "name": "超长查询",
            "query": "A" * 10000,
            "expected": ["过长", "截断", "限制"]
        },
        {
            "name": "特殊字符",
            "query": "!@#$%^&*()_+{}|:<>?",
            "expected": []
        },
        {
            "name": "SQL注入",
            "query": "'; DROP TABLE users; --",
            "expected": []
        },
        {
            "name": "Emoji",
            "query": "什么是CPU？🧠💻",
            "expected": []
        }]

    results = {
        "pass" : 0,
        "fail" : 0,
        "details":[]
    }

    for test in test_cases:
        print("=" * 20 + f"查询{test["name"]}" + "=" * 20)
        query = test['query']

        try:
            response = agent.invoke(
                {"messages":[HumanMessage(query)]},
                config
            )

            result = response['messages'][-1].content

            if result is not None:
                print(f"执行成功\n返回值为：{result}")

                error_keyword = any(kw in result for kw in ['error','failed','错误','失败'])
                if error_keyword:
                    print("返回了错误信息(但没崩溃)")
                    results["fail"] += 1
                else:
                    print("返回结果正确，执行成功")
                    results["pass"] += 1
            else:
                print("返回的结果是空值")
                results["fail"] += 1

        except Exception as e:
            print("+++程序破溃+++")
            print(f"崩溃类型{e}")
            results["fail"] += 1
            results["details"].append({
                "name" : test["name"],
                "error" : str(e)
            })

    print("=" * 20 + "测试结果总结" + "=" * 20)
    print(f"成功的次数：{results["pass"]}")
    print(f"失败的次数：{results["fail"]}")
    for detail in results["details"]:
        print(f"{detail["name"]}:{detail["error"]}")

    return results

test_error()


====================查询空查询====================
执行成功
返回值为：您好！我注意到您似乎没有输入具体的问题。😊

为了帮您从知识库中准确查找知识点，请您输入一个明确的问题。

**例如：**
- "什么是计算机网络？"
- "计算机发展经历了哪几个阶段？"
- "请解释操作系统的定义"

---

由于本次没有收到有效问题，暂时无法检索知识库并给出**置信度**。请您重新输入具体问题后，我会立即为您检索并附上置信度。
返回结果正确，执行成功
====================查询超长查询====================
执行成功
返回值为：您好！我注意到您输入的内容是一连串的字母 **"A"**，这似乎不是一个有效的问题。😊

为了帮您从知识库中准确查找知识点，请您输入一个**明确、具体的问题**。

---

### ✅ 正确提问示例
- "什么是计算机网络？"
- "计算机的发展经历了哪几个阶段？"
- "请解释操作系统的定义"
- "TCP/IP 协议有哪些层次？"

---

由于本次没有收到有效问题，我暂时无法检索知识库，也无法给出**置信度**。

请您重新输入具体问题后，我会立即为您检索并附上置信度。感谢您的理解！🙏
返回结果正确，执行成功
====================查询特殊字符====================
执行成功
返回值为：您好！我注意到您输入的内容是一串特殊符号（`!@#$%^&*()_+{}|:<>?`），这似乎不是一个有效的问题。😊

为了帮您从知识库中准确查找知识点，请您输入一个**明确、具体的问题**。

---

### ✅ 正确提问示例
- "什么是计算机网络？"
- "计算机的发展经历了哪几个阶段？"
- "请解释操作系统的定义"
- "TCP/IP 协议有哪些层次？"

---

由于本次没有收到有效问题，我暂时无法检索知识库，也无法给出**置信度**。

请您重新输入具体问题后，我会立即为您检索并附上置信度。感谢您的理解！🙏
返回结果正确，执行成功
====================查询SQL注入====================
执行成功
返回值为：您好！我注意到您输入的内容看起来像是一段 **SQL 注入尝试**（`'; DROP

{'pass': 5, 'fail': 0, 'details': []}

In [10]:
# ========================删除记忆========================
# 删除记忆
clean_thread("thread_2")

已删除 thread_id: thread_2


In [14]:
# 测试代码片段
def test_api():
    try:
        response = agent.invoke(
            {"messages": [HumanMessage("什么是应用层")]},  # 不触发工具调用
            config
        )
        print("✅ API 正常，余额充足")
        print(f"回复: {response['messages'][-1].content}")
    except Exception as e:
        print(f"❌ 错误: {e}")

test_api()

✅ API 正常，余额充足
回复: **信任度：58.49%**（置信度较低，请谨慎参考）

## 什么是应用层

应用层是计算机网络体系结构中的**最高层**。

**主要任务：**
- 通过应用进程间的交互来完成特定网络应用
- 应用层协议定义的是应用进程间通信和交互的规则（"进程"即主机中正在运行的程序）

**特点：**
- 不同的网络应用需要有不同的应用层协议
- 应用层交互的数据单元称为**报文（message）**

**常见应用层协议：**
- **DNS**：域名系统
- **HTTP**：支持万维网应用
- **SMTP**：支持电子邮件

> ⚠️ 本次检索的信任度仍为 **58.49%**（较低）。建议您更换更具体的询问方式，例如：
> - "应用层有哪些常见协议？"
> - "应用层的功能是什么？"
> - "应用层报文是什么？"
>
> 这样可以获得更高置信度的答案。


In [12]:
respond = vector_db.similarity_search_with_score("什么是应用层",k=2)
# raw_content = respond[0][0].page_content
print(respond)

[(Document(metadata={'total_pages': 464, 'creationdate': '2016-12-08T10:19:47+08:00', 'author': 'chen', 'producer': 'Adobe Acrobat Pro 11.0.0', 'page': 40, 'moddate': '2016-12-08T11:10:41+08:00', 'page_label': '41', 'creator': 'Adobe Acrobat Pro 11.0.0', 'source': './data/《计算机网络（第7版）》+谢希仁(1).pdf', 'title': ''}, page_content='应用层是体系结构中的最高层。应用层的任务是 通过应用进程间的交互来完成特定网\n络应用。应用层协议定义的是应用进程间通信和交互的规则。这里的进程就是指主机中正在\n运行的程序。对于不同的网络应用需要有不同的应用层协议。在互联网中的应用层协议很\n多，如域名系统 DNS，支持万维网应用的 HTTP 协议，支持电子邮件的 SMTP 协议，等\n等。我们把应用层交互的数据单元称为报文(message)。 \n(2) 运输层(transport layer) \n运输层的任务就是负责向 两台主机中进程之间的通信 提供通用的数据传输 服务。应用\n进程利用该服务传送应用层报文。所谓“通用的” ，是指并不针对某个特定网络应用，而是\n多种应用可以使用同一个运输层服务。由于一台主机可同时运行多个进程，因此运输层有复\n用和分用的功能。复用就是多个应用层进程可同时使用下面运输层的服务，分用和复用相'), 0.49891483783721924), (Document(metadata={'producer': 'Adobe Acrobat Pro 11.0.0', 'page': 260, 'source': './data/《计算机网络（第7版）》+谢希仁(1).pdf', 'creationdate': '2016-12-08T10:19:47+08:00', 'title': '', 'page_label': '261', 'creator': 'Adobe Acrobat Pro 11.0.0', 'author': 'chen'